## HW 6 Pt 2
Author: Minji Kang <br>
Date: 3/29/2025 <br>
Description: EDA with sub-goal- Explore if any college programs consistently grow players with above-average or below-average advanced NBA metrics (Duke, Kentucky, Chapel Hill, etc.). <br>

Used this article to help with mapping: https://medium.com/@alex_44314/use-python-geopandas-to-make-a-us-map-with-alaska-and-hawaii-39a9f5c222c6

In [2]:
# importing pandas for dataframes
import pandas as pd
# importing geopandas for mapping
import geopandas
import geopy
from geopy.geocoders import Nominatim
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Read in the merged dataset
mergedDF = pd.read_csv("https://raw.githubusercontent.com/hpatel-27/DataScience_CourseProject/refs/heads/main/datasets/merged_playerStats.csv", index_col=0)

# Read in players dataset
players = pd.read_csv("https://raw.githubusercontent.com/hpatel-27/DataScience_CourseProject/refs/heads/main/datasets/Players.csv", index_col=0)


In [3]:
mergedDF.columns

Index(['height', 'weight', 'college', 'Year', 'Pos', 'Age', 'G', 'MP', 'PER',
       'TS%', 'USG%', 'WS', 'BPM', 'VORP', 'FG%', '3P%', '2P%', 'eFG%', 'FT%',
       'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PTS'],
      dtype='object')

In [4]:
players.columns

Index(['Player', 'height', 'weight', 'collage', 'born', 'birth_city',
       'birth_state'],
      dtype='object')

In [5]:
# getting birth location from original players dataset since it was dropped for merged dataset
locationDF = players[['birth_city', 'birth_state']]
# setting index as player name
locationDF.index = players['Player']
# adding advanced metrics and college name
locationDF = locationDF.join(mergedDF[['college', 'BPM', 'VORP', 'WS']])
locationDF

,birth_city,birth_state,college,BPM,VORP,WS
Player,,,,,,
Curly Armstrong,NaN,NaN,Indiana University,NaN,NaN,2.408911
Cliff Barker,Yorktown,Indiana,University of Kentucky,NaN,NaN,0.677852
Leo Barnhorst,NaN,NaN,University of Notre Dame,NaN,NaN,NaN
Ed Bartels,NaN,NaN,North Carolina State University,NaN,NaN,-0.538298
Ralph Beard,Hardinsburg,Kentucky,University of Kentucky,NaN,NaN,5.690476
...,...,...,...,...,...,...
Troy Williams,Columbia,South Carolina,South Carolina State University,-2.76,-0.09,0.180000
Kyle Wiltjer,Portland,Oregon,Gonzaga University,-4.00,0.00,0.000000
Stephen Zimmerman,Hendersonville,Tennessee,"University of Nevada, Las Vegas",-7.30,-0.10,0.000000


In [6]:
# dropping row if birth location is not available
birthDF = locationDF.dropna(subset=['birth_state'])
birthDF

,birth_city,birth_state,college,BPM,VORP,WS
Player,,,,,,
Cliff Barker,Yorktown,Indiana,University of Kentucky,NaN,NaN,0.677852
Ralph Beard,Hardinsburg,Kentucky,University of Kentucky,NaN,NaN,5.690476
Charlie Black,Arco,Idaho,University of Kansas,NaN,NaN,1.877622
Nelson Bobb,Philadelphia,Pennsylvania,Temple University,NaN,NaN,1.521145
Jake Bornheimer,New Brunswick,New Jersey,Muhlenberg College,NaN,NaN,0.800000
...,...,...,...,...,...,...
Troy Williams,Columbia,South Carolina,South Carolina State University,-2.76,-0.09,0.180000
Kyle Wiltjer,Portland,Oregon,Gonzaga University,-4.00,0.00,0.000000
Stephen Zimmerman,Hendersonville,Tennessee,"University of Nevada, Las Vegas",-7.30,-0.10,0.000000


In [7]:
# dropping row if college is not available
collegeDF = locationDF.dropna(subset=['college'])
collegeDF

,birth_city,birth_state,college,BPM,VORP,WS
Player,,,,,,
Curly Armstrong,NaN,NaN,Indiana University,NaN,NaN,2.408911
Cliff Barker,Yorktown,Indiana,University of Kentucky,NaN,NaN,0.677852
Leo Barnhorst,NaN,NaN,University of Notre Dame,NaN,NaN,NaN
Ed Bartels,NaN,NaN,North Carolina State University,NaN,NaN,-0.538298
Ralph Beard,Hardinsburg,Kentucky,University of Kentucky,NaN,NaN,5.690476
...,...,...,...,...,...,...
Okaro White,Clearwater,Florida,Florida State University,-2.10,0.00,0.600000
Isaiah Whitehead,Brooklyn,New York,Seton Hall University,-4.90,-1.20,-0.800000
Troy Williams,Columbia,South Carolina,South Carolina State University,-2.76,-0.09,0.180000


In [21]:
# connecting to github to get folder with US Census map data
check = !git status
if 'main' not in check[0]:
  !git clone https://github.com/hpatel-27/DataScience_CourseProject.git
else if 'mkang8' not in check[0]:
  !git checkout mkang8
  %cd DataScience_CourseProject/
else:
  print('Already connected to GitHub')

SyntaxError: expected ':' (<ipython-input-21-d0ad14583541>, line 5)

In [20]:
check

['On branch main',
 "Your branch is up to date with 'origin/main'.",
 '',
 'nothing to commit, working tree clean']

In [15]:
!ls

datasets  dictionaries	meeting_minutes  project.ipynb	README.md  stats_explained.txt


In [16]:
# reading folder into geopandas
gdf = geopandas.read_file("cb_2018_us_state_500k")
# merging datasets using state name
gdf = gdf.merge(birthDF, left_on='NAME', right_on='birth_state', how='inner')
gdf.head()

DataSourceError: cb_2018_us_state_500k: No such file or directory

In [ ]:
# filter if state is Hawaii or Alaska
print('Hawaii: ' + str(birthDF[birthDF['birth_state'] == 'Hawaii'].shape[0]))
print('Alaska: ' + str(birthDF[birthDF['birth_state'] == 'Alaska'].shape[0]))

In [ ]:
# omitting Hawaii and Alaska for easier graphing
gdfMain = gdf[gdf['NAME'] != 'Hawaii']
gdfMain = gdfMain[gdfMain['NAME'] != 'Alaska']

In [ ]:
countByBirth = gdfMain[['NAME', 'birth_state']].groupby('birth_state').count().sort_values(by='NAME', ascending=False)

In [ ]:
norm = mcolors.Normalize(countByBirth['NAME'].min(), countByBirth['NAME'].max())
mapper = plt.cm.ScalarMappable(norm=norm, cmap='Blues')
gdfMain['birth_state_color'] = countByBirth['NAME'].apply(lambda x: mcolors.to_hex(mapper.to_rgba(x)))

In [ ]:
# creating map for birth state
fig, ax = plt.subplots(1, figsize=(10, 6))
gdfMain.plot(column='birth_state_color', cmap='YlOrRd', linewidth=0.8, ax=ax, edgecolor='0.8', legend=True)
ax.axis('off')
